<a href="https://colab.research.google.com/github/ismael-almazan/Ingenier-a-de-datos-avanzada/blob/main/Practica_SQL_Fintech_DML_263177.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 📘 Práctica SQL – Fintech Databank

Este notebook contiene los **ejercicios prácticos** sobre la base de datos `fintech_databank.db`.  
Tu objetivo será escribir consultas SQL para resolver cada problema.

👉 **Recuerda:** conecta la base de datos antes de empezar:
```python
%sql sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
```


**<h1>MAESTRIA EN INTELIGENCIA ARTIFICIAL Y ANALITICA DE DATOS <h1>**

<h2>MATERIA: INGENIERIA DE DATOS AVANZADA<h2>

<h3>DOCENTE: VICENTE GARCIA JIMENEZ <h3>

**<h3>ALUMNO: ISMAEL ALMAZAN LUNA<h3>**

FECHA: 06/02/2025

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
!pip install sqlalchemy
!pip install ipython-sql

%load_ext sql
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

%sql sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db


## 🔹 Instrucciones generales
1. Conéctate a la base de datos.  
2. Antes de cada ejercicio, revisa las tablas disponibles con:
   ```sql
   %%sql
   SELECT name FROM sqlite_master WHERE type='table';
   ```
3. Ejecuta tus consultas SQL en las celdas de código marcadas como `%%sql`.  
4. Guarda tus respuestas en este notebook.


In [26]:
%%sql
SELECT name FROM sqlite_master WHERE type='table';

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


name
clientes
cuentas
transacciones
tipos_cambio


## Ejercicio 1. Exploración básica


- Lista todas las filas de la tabla **clientes**.  
- Muestra únicamente el nombre y la ciudad de cada cliente.  
- ¿Cuántos clientes hay en total?  


In [27]:
#Tabla de clientes
%%sql
SELECT *
FROM clientes;

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


cliente_id,nombre,ciudad,segmento
C001,Ana Torres,CDMX,Retail
C002,Luis Gómez,Guadalajara,PYME
C003,Textiles J&R,Saltillo,PYME


In [29]:
#Nombre y ciudad de cada cliente
%%sql
SELECT nombre, ciudad
FROM clientes;

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


nombre,ciudad
Ana Torres,CDMX
Luis Gómez,Guadalajara
Textiles J&R,Saltillo


In [30]:
#Total de clientes
%%sql
SELECT COUNT(*) as Total_clientes
FROM clientes;

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


Total_clientes
3


## Ejercicio 2. Cuentas de clientes


- Obtén todas las cuentas del cliente **C001**.  
- Lista las cuentas que están en moneda **USD**.  
- Muestra el número de cuentas que tiene cada cliente (usa `GROUP BY`).  


In [24]:
# Cuentas del cliente C001
%%sql
SELECT * FROM cuentas
WHERE cliente_id == "C001";

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


cuenta_id,cliente_id,moneda,tipo,fecha_apertura
A1001,C001,MXN,Vista,2024-11-10
A1002,C001,USD,Ahorro,2024-12-01


In [27]:
#Cuentas en USd
%%sql
SELECT * FROM cuentas
WHERE moneda == "USD";

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


cuenta_id,cliente_id,moneda,tipo,fecha_apertura
A1002,C001,USD,Ahorro,2024-12-01


In [43]:
#Numero de cuentas de cada cliente
%%sql
SELECT cliente_id, COUNT(*) AS Numero_de_cuentas
FROM cuentas
GROUP BY cliente_id;

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


cliente_id,Numero_de_cuentas
C001,2
C002,1
C003,1


## Ejercicio 3. Transacciones


- Obtén todas las transacciones de la cuenta **A1001**.  
- Calcula el **saldo neto** de esa cuenta sumando la columna `monto`.  
- Lista las transacciones que sean **comisiones**.  


In [44]:
%%sql
SELECT *
FROM transacciones
WHERE cuenta_id == "A1001";

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


tx_id,cuenta_id,fecha,concepto,monto,categoria
T0001,A1001,2025-07-01,Nómina,25000,Depósito
T0002,A1001,2025-07-03,Renta,-12000,Pago
T0003,A1001,2025-07-05,Super,-2200,Pago
T0009,A1001,2025-07-15,Comisión retiro,-35,Comisión


In [48]:
#Saldo neto de A1001
%%sql
SELECT cuenta_id, SUM(monto) AS Saldo_neto
FROM transacciones
WHERE cuenta_id == "A1001";

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


cuenta_id,Saldo_neto
A1001,10765


In [32]:
#Comisiones
%%sql
SELECT *
FROM transacciones
WHERE categoria == "Comisión";

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


tx_id,cuenta_id,fecha,concepto,monto,categoria
T0009,A1001,2025-07-15,Comisión retiro,-35,Comisión


## Ejercicio 4. Reportes con agrupaciones


- Calcula el total de transacciones (suma de `monto`) por **categoría**.     


In [6]:
%%sql
SELECT categoria, SUM(monto) AS total_transacciones
FROM transacciones
GROUP BY categoria;

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


categoria,total_transacciones
Comisión,-80
Depósito,25800
Interés,123.5
Pago,-82200
Transferencia,95000



## Ejercicio 5. **Modificar datos (UPDATE)**

> **Objetivo:** actualizar valores existentes usando `WHERE` de forma segura.

1. Cambia el `segmento` del cliente **C002** de *Retail* a **PYME**.  
2. Corrige la `ciudad` de **Textiles J&R (C003)** a **Saltillo**.  


> **Tip:** siempre valida primero con un `SELECT` y después aplica el `UPDATE`.


In [33]:
# 1.
#%sql SELECT * FROM clientes
%%sql
UPDATE clientes
SET segmento = "PYME"
WHERE cliente_id = "C002";

%sql SELECT * FROM clientes;

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


cliente_id,nombre,ciudad,segmento
C001,Ana Torres,CDMX,Retail
C002,Luis Gómez,Guadalajara,PYME
C003,Textiles J&R,Saltillo,PYME


In [36]:
# 2.
%sql SELECT * FROM clientes;

%%sql
UPDATE clientes
SET ciudad = "Saltillo"
WHERE cliente_id = "C003";

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


cliente_id,nombre,ciudad,segmento
C001,Ana Torres,CDMX,Retail
C002,Luis Gómez,Guadalajara,PYME
C003,Textiles J&R,Saltillo,PYME



## Ejercicio 6. **Borrar datos (DELETE)**

> **Objetivo:** eliminar filas específicas manteniendo la integridad y usando `WHERE`.

1. Elimina **solo** las transacciones de la cuenta **A3001** del día **2025-07-12**.  
2. Elimina la **comisión** de `A2001` del **2025-07-04**.  
> **Tip:** prueba primero con `SELECT` para verificar el subconjunto a borrar.


In [20]:
# 1.
#%sql SELECT * FROM transacciones;

%%sql
DELETE FROM transacciones
WHERE cuenta_id = "A3001" AND fecha = "2025-07-12";

%sql SELECT * FROM transacciones;

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


tx_id,cuenta_id,fecha,concepto,monto,categoria
T0001,A1001,2025-07-01,Nómina,25000,Depósito
T0002,A1001,2025-07-03,Renta,-12000,Pago
T0003,A1001,2025-07-05,Super,-2200,Pago
T0004,A1002,2025-07-07,Ahorro mensual,800,Depósito
T0005,A2001,2025-07-02,Interés mensual,120,Interés
T0007,A3001,2025-07-10,Pago nómina empleados,-68000,Pago
T0009,A1001,2025-07-15,Comisión retiro,-35,Comisión
T0010,A1002,2025-07-20,Interés mensual,3.5,Interés


In [21]:
# 2.
#%sql SELECT * FROM transacciones;

%%sql
DELETE FROM transacciones
WHERE cuenta_id = "A2001" AND fecha = "2025-07-04";

%sql SELECT * FROM transacciones;

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


tx_id,cuenta_id,fecha,concepto,monto,categoria
T0001,A1001,2025-07-01,Nómina,25000,Depósito
T0002,A1001,2025-07-03,Renta,-12000,Pago
T0003,A1001,2025-07-05,Super,-2200,Pago
T0004,A1002,2025-07-07,Ahorro mensual,800,Depósito
T0005,A2001,2025-07-02,Interés mensual,120,Interés
T0007,A3001,2025-07-10,Pago nómina empleados,-68000,Pago
T0009,A1001,2025-07-15,Comisión retiro,-35,Comisión
T0010,A1002,2025-07-20,Interés mensual,3.5,Interés
